# 🎙️ OpenVoice V1 & V2 Batch Voice Cloning - Google Colab T4

Notebook ini khusus untuk memproses **OpenVoice V1** dan **OpenVoice V2** secara otomatis di Google Colab T4.

### 📌 Pilihan Versi OpenVoice:
- **OpenVoice V1**: Menggunakan checkpoint `myshell-ai/OpenVoice` (V1 Converter).
- **OpenVoice V2**: Menggunakan checkpoint `myshell-ai/OpenVoice-V2` (V2 Converter & MeloTTS).

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# STEP 2: Install System & Python Dependencies
!apt-get update -q
!apt-get install -y -q mecab libmecab-dev mecab-ipadic-utf8

!pip install --upgrade pip setuptools wheel
!pip install Cython numpy
!pip install mecab-python3 unidic num2words pythainlp subword_nmt cached_path cn2an pypinyin pykakasi g2p_en g2pM fugashi ipadic unidic_lite anyascii jamo gruut gruut_ipa huggingface_hub
!python -m unidic download

# Clone & Install MeloTTS dan OpenVoice
!pip install --no-deps git+https://github.com/myshell-ai/MeloTTS.git
%cd /content
!rm -rf OpenVoice
!git clone --depth 1 https://github.com/myshell-ai/OpenVoice.git
%cd /content/OpenVoice
!pip install --no-deps -e .


In [ ]:
# STEP 3: Download OpenVoice V1 & V2 Checkpoints
import os, urllib.request, zipfile
from pathlib import Path
from huggingface_hub import snapshot_download

repo = Path('/content/OpenVoice')

# Download OpenVoice V1 Checkpoints
ckpt_v1 = repo / 'checkpoints'
if not (ckpt_v1 / 'converter' / 'checkpoint.pth').exists():
    print('📥 Downloading OpenVoice V1 Checkpoints...')
    snapshot_download(repo_id='myshell-ai/OpenVoice', local_dir=repo, local_dir_use_symlinks=False)
    print('✅ Checkpoints OpenVoice V1 Siap.')
else:
    print('✅ Checkpoints OpenVoice V1 Sudah Ada.')

# Download OpenVoice V2 Checkpoints
ckpt_v2 = repo / 'checkpoints_v2'
if not (ckpt_v2 / 'converter' / 'checkpoint.pth').exists():
    print('📥 Downloading OpenVoice V2 Checkpoints...')
    zip_path = repo / 'checkpoints_v2_0417.zip'
    url = 'https://huggingface.co/cqchangm/openvoice2/resolve/main/checkpoints_v2_0417.zip'
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(repo)
    zip_path.unlink(missing_ok=True)
    print('✅ Checkpoints OpenVoice V2 Siap.')
else:
    print('✅ Checkpoints OpenVoice V2 Sudah Ada.')


In [ ]:
# STEP 4: Konfigurasi Batch & Pilihan Versi OpenVoice
RUN_OPENVOICE_V1 = True   # Set True untuk menjalankan OpenVoice V1
RUN_OPENVOICE_V2 = True   # Set True untuk menjalankan OpenVoice V2
SKIP_EXISTING = True      # Skip jika audio sudah ada di Drive
BATCH_START = 0
BATCH_COUNT = 100

TEXTS = {
    'id': 'Indonesia adalah negara kepulauan yang kaya budaya, bahasa, sumber daya alam, serta memiliki masyarakat beragam yang hidup saling berdampingan harmonis.',
    'en': 'England is a historic European country known for its royal family, famous universities, football, diverse culture, and remarkable technological development worldwide.'
}
print(f'⚙️ Konfigurasi: V1={RUN_OPENVOICE_V1}, V2={RUN_OPENVOICE_V2}')


In [ ]:
# STEP 5: Run OpenVoice V1 & V2 Batch Engine
import os, sys, gc, shutil, subprocess
from pathlib import Path
import torch
import nltk

nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('cmudict', quiet=True)

from melo.api import TTS
from openvoice import se_extractor
from openvoice.api import ToneColorConverter

ROOT = Path('/content')
REF_ROOT = ROOT / 'reference_dataskripsi'
BASE_DRIVE_OUT = Path('/content/drive/MyDrive/data skirpsi/VoiceCloning')
REF_ROOT.mkdir(parents=True, exist_ok=True)
BASE_DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def run_cmd(*args):
    cmd = [str(a) for a in args]
    return subprocess.run(cmd, capture_output=True, text=True)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def scan_references():
    drive_folder = Path('/content/drive/MyDrive/Dataskripsi')
    if not drive_folder.exists():
        cands = list(Path('/content/drive/MyDrive').rglob('*Dataskripsi*'))
        if cands: drive_folder = cands[0]
        else: raise FileNotFoundError('Folder Dataskripsi tidak ditemukan di Google Drive!')
    
    print(f'📂 Scanning folder reference dari: {drive_folder}')
    audio_exts = {'.wav', '.mp3', '.m4a', '.ogg', '.flac'}
    copied = 0
    for f in drive_folder.rglob('*'):
        if f.is_file() and f.suffix.lower() in audio_exts:
            target_name = f.stem + '.wav' if f.suffix.lower() != '.wav' else f.name
            target_path = REF_ROOT / target_name
            if not target_path.exists():
                if f.suffix.lower() != '.wav':
                    run_cmd('ffmpeg', '-y', '-i', str(f), '-ar', '24000', '-ac', '1', str(target_path))
                else:
                    shutil.copy2(f, target_path)
                copied += 1
    print(f'  ✅ Prepared {copied} new audio file(s) di {REF_ROOT}')
    
    speaker_map = {}
    for p in REF_ROOT.glob('*.wav'):
        name = p.stem
        spk_id = name
        lang = 'id'
        if name.startswith('Suaraindo_'):
            spk_id = name[len('Suaraindo_'):]
            lang = 'id'
        elif name.startswith('Suaraeng_'):
            spk_id = name[len('Suaraeng_'):]
            lang = 'en'
        elif '_' in name:
            spk_id = name.split('_')[-1]
        
        if spk_id not in speaker_map: speaker_map[spk_id] = {'id': p, 'en': p}
        if lang == 'id': speaker_map[spk_id]['id'] = p
        elif lang == 'en': speaker_map[spk_id]['en'] = p
    return speaker_map

spk_map = scan_references()
all_speakers = sorted(list(spk_map.keys()))
selected_speakers = all_speakers[BATCH_START : BATCH_START + BATCH_COUNT]
print(f'🎙️ Total Speaker Terdeteksi: {len(all_speakers)} | Memproses Batch ({len(selected_speakers)} speaker): {selected_speakers}')
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
repo = Path('/content/OpenVoice')

# ============================================================
# 1. RUN OPENVOICE V1 BATCH
# ============================================================
if RUN_OPENVOICE_V1:
    print('\n==================== MODEL: OpenVoice V1 ====================')
    v1_dir = repo / 'checkpoints' / 'converter'
    if (v1_dir / 'config.json').exists():
        converter_v1 = ToneColorConverter(str(v1_dir / 'config.json'), device=device)
        converter_v1.load_ckpt(str(v1_dir / 'checkpoint.pth'))
        
        base_tts = TTS(language='EN', device=device)
        speaker_id = next(iter(base_tts.hps.data.spk2id.values()))
        source_se_path = repo / 'checkpoints' / 'base_speakers' / 'ses' / 'en-default.pth'
        if not source_se_path.exists():
            source_se_path = repo / 'checkpoints' / 'base_speakers' / 'ses' / 'en-us.pth'
        source_se_v1 = torch.load(str(source_se_path), map_location=device)
        
        ov1_out = ROOT / 'generated_voice_cloning' / 'OpenVoice_V1'
        ov1_drive = BASE_DRIVE_OUT / 'OpenVoice_V1'
        ov1_out.mkdir(parents=True, exist_ok=True)
        ov1_drive.mkdir(parents=True, exist_ok=True)
        
        for spk in selected_speakers:
            ref_id = spk_map[spk]['id']
            ref_en = spk_map[spk]['en']
            try:
                target_id, _ = se_extractor.get_se(str(ref_id), converter_v1, vad=True, target_dir=str(ROOT / 'processed'))
                target_en, _ = se_extractor.get_se(str(ref_en), converter_v1, vad=True, target_dir=str(ROOT / 'processed'))
                
                for lang, text, target in (('id', TEXTS['id'], target_id), ('en', TEXTS['en'], target_en)):
                    final_mp3_name = f'{spk}_{lang}.mp3'
                    drive_mp3 = ov1_drive / final_mp3_name
                    if SKIP_EXISTING and drive_mp3.exists() and drive_mp3.stat().st_size > 1000:
                        print(f'  ⏩ [SKIP] OpenVoice V1 {spk}_{lang} (Sudah ada di Drive)')
                        continue
                    
                    print(f'  ⚡ [GENERATE] OpenVoice V1 {spk}_{lang}...')
                    tmp_wav = ov1_out / f'tmp_{spk}_{lang}.wav'
                    final_wav = ov1_out / f'{spk}_{lang}.wav'
                    final_mp3 = ov1_out / final_mp3_name
                    
                    base_tts.tts_to_file(text, speaker_id, str(tmp_wav), speed=1.0)
                    converter_v1.convert(str(tmp_wav), source_se_v1, target, str(final_wav), message='@Research')
                    run_cmd('ffmpeg', '-y', '-i', str(final_wav), '-b:a', '192k', str(final_mp3))
                    shutil.copy2(final_mp3, drive_mp3)
                    print(f'  ☁️ Saved to Drive: {final_mp3_name}')
                    if tmp_wav.exists(): tmp_wav.unlink()
            except Exception as e:
                print(f'  ❌ [ERROR] OpenVoice V1 {spk}: {e}')
            finally:
                free_memory()
        
        del converter_v1, base_tts, source_se_v1
        free_memory()
    else:
        print('❌ Checkpoint OpenVoice V1 tidak ditemukan!')

# ============================================================
# 2. RUN OPENVOICE V2 BATCH
# ============================================================
if RUN_OPENVOICE_V2:
    print('\n==================== MODEL: OpenVoice V2 ====================')
    v2_dir = repo / 'checkpoints_v2' / 'converter'
    if (v2_dir / 'config.json').exists():
        converter_v2 = ToneColorConverter(str(v2_dir / 'config.json'), device=device)
        converter_v2.load_ckpt(str(v2_dir / 'checkpoint.pth'))
        
        base_tts = TTS(language='EN', device=device)
        speaker_id = next(iter(base_tts.hps.data.spk2id.values()))
        source_se_v2 = torch.load(str(repo / 'checkpoints_v2' / 'base_speakers' / 'ses' / 'en-us.pth'), map_location=device)
        
        ov2_out = ROOT / 'generated_voice_cloning' / 'OpenVoice_V2'
        ov2_drive = BASE_DRIVE_OUT / 'OpenVoice_V2'
        ov2_out.mkdir(parents=True, exist_ok=True)
        ov2_drive.mkdir(parents=True, exist_ok=True)
        
        for spk in selected_speakers:
            ref_id = spk_map[spk]['id']
            ref_en = spk_map[spk]['en']
            try:
                target_id, _ = se_extractor.get_se(str(ref_id), converter_v2, vad=True, target_dir=str(ROOT / 'processed'))
                target_en, _ = se_extractor.get_se(str(ref_en), converter_v2, vad=True, target_dir=str(ROOT / 'processed'))
                
                for lang, text, target in (('id', TEXTS['id'], target_id), ('en', TEXTS['en'], target_en)):
                    final_mp3_name = f'{spk}_{lang}.mp3'
                    drive_mp3 = ov2_drive / final_mp3_name
                    if SKIP_EXISTING and drive_mp3.exists() and drive_mp3.stat().st_size > 1000:
                        print(f'  ⏩ [SKIP] OpenVoice V2 {spk}_{lang} (Sudah ada di Drive)')
                        continue
                    
                    print(f'  ⚡ [GENERATE] OpenVoice V2 {spk}_{lang}...')
                    tmp_wav = ov2_out / f'tmp_{spk}_{lang}.wav'
                    final_wav = ov2_out / f'{spk}_{lang}.wav'
                    final_mp3 = ov2_out / final_mp3_name
                    
                    base_tts.tts_to_file(text, speaker_id, str(tmp_wav), speed=1.0)
                    converter_v2.convert(str(tmp_wav), source_se_v2, target, str(final_wav), message='@Research')
                    run_cmd('ffmpeg', '-y', '-i', str(final_wav), '-b:a', '192k', str(final_mp3))
                    shutil.copy2(final_mp3, drive_mp3)
                    print(f'  ☁️ Saved to Drive: {final_mp3_name}')
                    if tmp_wav.exists(): tmp_wav.unlink()
            except Exception as e:
                print(f'  ❌ [ERROR] OpenVoice V2 {spk}: {e}')
            finally:
                free_memory()
        
        del converter_v2, base_tts, source_se_v2
        free_memory()
    else:
        print('❌ Checkpoint OpenVoice V2 tidak ditemukan!')

print('\n🎉 OPENVOICE GENERATION SELESAI!')
